# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirl0w/Machine-Learning-Intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


For my lane (Refresh/Content Opportunity Scoring), I'm choosing a Decision Tree,
then comparing it against a Random Forest.

Why: my Week-4 baseline is a hand-written rule combining staleness, visibility,
and CTR. A decision tree is the natural next step — still interpretable (I can
print and read it, same as Notebook 02), but able to discover better threshold
combinations than a human-picked rule. Random Forest is added as a stronger
comparison to check whether extra complexity is actually worth it, per the
"does not reward complexity alone" requirement.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Split design: client-holdout (grouped) split — pages from the same client never
appear in both train and test. This matches the real deployment scenario: the
model will eventually be judged on clients it has never seen.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(X_train)} rows, {df['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test)} rows, {df['client_id'].iloc[test_idx].nunique()} clients")

Train: 23837 rows, 25 clients
Test:  6163 rows, 7 clients


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

test_df = df.iloc[test_idx].copy()
test_df["baseline_score"] = (
    ((test_df["days_since_last_update"] >= 180) & (test_df["impressions_90d"] >= 500)).astype(int) * test_df["impressions_90d"] * 0.5
    + ((test_df["ctr"] < 0.3) & (test_df["impressions_90d"] >= 500)).astype(int) * test_df["impressions_90d"] * 0.5
)

tree = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:,1]

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:,1]

results = pd.DataFrame({
    "method": ["Baseline (Week 4 rule)", "Decision Tree", "Random Forest"],
    "Precision@20": [
        precision_at_k(test_df["baseline_score"], y_test.values, 20),
        precision_at_k(tree_scores, y_test.values, 20),
        precision_at_k(rf_scores, y_test.values, 20),
    ],
    "Precision@50": [
        precision_at_k(test_df["baseline_score"], y_test.values, 50),
        precision_at_k(tree_scores, y_test.values, 50),
        precision_at_k(rf_scores, y_test.values, 50),
    ],
})
print(results)

                   method  Precision@20  Precision@50
0  Baseline (Week 4 rule)          0.45          0.44
1           Decision Tree          0.55          0.60
2           Random Forest          0.80          0.72


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

# Look at false positives: flagged as declining, but actually not
test_df["rf_score"] = rf_scores
false_positives = test_df[(test_df["rf_score"] > 0.5) & (test_df["is_declining_label"] == 0)]
print(f"\nFalse positives: {len(false_positives)} rows")
false_positives[["impressions_90d","days_since_last_update","ctr","avg_position"]].describe()

impressions_90d           0.275838
avg_position              0.247979
content_age_days          0.161297
word_count                0.159195
ctr                       0.118977
days_since_last_update    0.036715
dtype: float64

False positives: 1469 rows


,impressions_90d,days_since_last_update,ctr,avg_position
count,1469.000000,1469.000000,1469.000000,1469.000000
mean,5502.714091,39.665078,0.237733,16.569231
std,17909.674679,36.770120,0.678503,15.082691
min,1.000000,7.000000,0.000000,1.000000
25%,66.000000,20.000000,0.000000,7.300000
50%,783.000000,20.000000,0.090000,11.100000
75%,3799.000000,89.000000,0.280000,19.700000
max,345111.000000,106.000000,20.000000,85.500000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.